# 03 — Measuring overlap: IoU, ProbIoU, and box losses

Lesson 02 compared boxes with exact rotated IoU. Training needs more: a loss that is differentiable, behaves at the angle boundary, and is cheap on a GPU. This lesson puts the box losses OrientedDet ships side by side, all in pure PyTorch, then maps each one to the detector and config key that uses it.

| | |
|---|---|
| Needs | Only the install |
| Kaggle | Yes (CPU is fine) |
| Posts | [Rotated Faster R-CNN on DOTA without custom CUDA](https://deeplearning.earth/posts/2026-07-10_rotated_faster_rcnn_probiou_dota/) · [v0.2.0: Rotated FCOS and decoded rIoU](https://deeplearning.earth/posts/2026-08-28_oriented-det_v0_2_0_rotated_fcos_decoded_riou_and_the_updated_zoo/) |

## The options

- **Smooth L1 / L1 on encoded offsets** `(dx, dy, log dw, log dh, dθ)`. Simple and stable, but the five numbers are penalised independently, so a correct box in a different spelling (lesson 02) still costs loss.
- **Exact rotated IoU** (polygon clipping, `rbox_iou`). The metric itself, but classic implementations are CUDA kernels. OrientedDet uses it for evaluation.
- **Differentiable rotated IoU** (`diff_iou_rotated_2d`), trained as the *decoded rIoU* loss `1 − IoU` on decoded boxes. Pure-PyTorch polygon intersection with gradients. Used by Rotated FCOS and RTMDet.
- **Gaussian surrogates.** Turn each box into a 2-D Gaussian (mean = centre, covariance from `w`, `h`, `θ`) and compare the distributions. **ProbIoU** uses the Bhattacharyya distance. **KFIoU** approximates overlap the way a Kalman filter fuses two estimates. Both are smooth, cheap, and periodic in θ by construction.

## Setup

The same two cells open every lesson. The first installs OrientedDet: your editable checkout if this notebook sits inside the repo, otherwise the PyPI wheel. The second sets paths and the device, and defines the small helpers used across the series (`hub_files`, `hub_model`, `detect`, `show`, ...). On Kaggle, switch **Internet ON** in the notebook settings.

In [ ]:
# Install: editable checkout when this notebook lives inside the repo, else PyPI.
import importlib.util
from pathlib import Path

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file() and (p / "oriented_det").is_dir()), None)
_spec = importlib.util.find_spec("oriented_det")
if REPO is None:
    %pip install -q "oriented-det>=0.3.1"
elif _spec is None or not Path(_spec.origin).resolve().is_relative_to(REPO.resolve()):
    %pip install -q -e "{REPO}"

In [ ]:
import os, subprocess, sys, time, urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import oriented_det

GH_RAW = "https://raw.githubusercontent.com/DL4EO/oriented-det/main"
IN_KAGGLE = Path("/kaggle/working").is_dir()
HAS_REPO = REPO is not None
WORK = (Path("/kaggle/working") if IN_KAGGLE else
        Path(os.environ.get("ORIENTED_DET_TUTORIAL_WORK", (REPO or Path.cwd()) / "work_dirs")).resolve())
WORK.mkdir(parents=True, exist_ok=True)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK)  # runs/ and predictions/ land here
if not HAS_REPO:  # wheel install: keep Hub weights out of site-packages
    os.environ.setdefault("ORIENTED_DET_PRETRAINED_DIR", str(WORK / "pretrained"))

CONFIGS = Path(oriented_det.__file__).parent / "configs"  # recipes shipped with the package
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
ODET = [sys.executable, "-m", "oriented_det.cli"]  # same as typing `odet` in a shell


def run(cmd):
    """Run a CLI command, echoing it first."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(["odet", *cmd[3:]] if cmd[:3] == ODET else cmd))
    subprocess.run(cmd, check=True)


def demo_file(rel):
    """demo/<rel> from the checkout, else downloaded once from GitHub into WORK/demo."""
    if HAS_REPO and (REPO / "demo" / rel).is_file():
        return REPO / "demo" / rel
    target = WORK / "demo" / rel
    if not target.is_file():
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{GH_RAW}/demo/{rel}", target)
    return target


def kaggle_input(*needles):
    """First /kaggle/input/<dataset> whose name contains one of `needles` (None off Kaggle)."""
    root = Path("/kaggle/input")
    dirs = sorted(p for p in root.iterdir() if p.is_dir()) if root.is_dir() else []
    return next((p for n in needles for p in dirs if n.lower() in p.name.lower()), None)


def hub_files(slug):
    """Download a Hub checkpoint and its sidecar config once -> (checkpoint_path, config_path)."""
    from huggingface_hub import hf_hub_download
    from oriented_det.pretrained import ensure_checkpoint, load_manifest, resolve_checkpoint_sidecar_config

    ckpt = ensure_checkpoint(f"hf://{slug}")
    manifest = load_manifest()
    sidecar = resolve_checkpoint_sidecar_config(ckpt) or hf_hub_download(
        manifest["repo_id"], ckpt.with_suffix(".json").name,
        revision=manifest.get("revision", "main"), local_dir=ckpt.parent)
    return ckpt, Path(sidecar)


def hub_model(slug, device=DEVICE):
    """Hub checkpoint -> (model, config, class_names). Lesson 01 unpacks it."""
    from oriented_det.runtime.checkpoint import load_model_from_checkpoint

    ckpt, sidecar = hub_files(slug)
    return load_model_from_checkpoint(str(ckpt), str(sidecar), device=device)


def detect(model, config, class_names, image, score_thr=None, classes=None):
    """Detections on a PIL image with the checkpoint's deploy settings (sliding windows past the canvas)."""
    from oriented_det.runtime.inference import apply_nms_to_detections, run_inference_auto
    from oriented_det.train.config import (get_preprocessing_params, resolve_inference_score_threshold,
                                           resolve_inference_sliding_window_overlap_pixels)

    nms = config.production.final_nms_iou_threshold
    dets = run_inference_auto(
        image=image, model=model, device=DEVICE, preprocessing=get_preprocessing_params(config),
        score_threshold=resolve_inference_score_threshold(config) if score_thr is None else score_thr,
        nms_threshold=nms, overlap_pixels=resolve_inference_sliding_window_overlap_pixels(config),
        class_names=class_names)
    dets = apply_nms_to_detections(dets, iou_threshold=nms)
    if classes:
        dets = [d for d in dets if class_names[int(d["label"]) - 1] in classes]
    return dets


def show(*images, titles=(), size=6):
    """Display images side by side."""
    _, axes = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for ax, img, title in zip(axes[0], images, [*titles, *[""] * len(images)]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print(f"oriented_det: {Path(oriented_det.__file__).parent}")
print(f"device={DEVICE}  kaggle={IN_KAGGLE}  work={WORK}")

## 1. Sweep the angle

Fix a 140×40 target at 30° and rotate an otherwise perfect prediction away from it. Each curve is a *loss*, so lower is better. KFIoU's overlap ratio tops out at 1/3 by construction, so it is rescaled to `1 − 3·ratio`.

In [ ]:
import math
import numpy as np
from oriented_det.geometry import RBox
from oriented_det.ops import diff_iou_rotated_2d, kfiou_overlap_ratio, probiou_loss_per_box, rbox_iou

target = torch.tensor([[100.0, 100.0, 140.0, 40.0, math.radians(30)]])
offsets = np.arange(0, 180, 0.5)  # stops short of exactly 180 deg, where corners coincide
preds = target.repeat(len(offsets), 1)
preds[:, 4] += torch.tensor(np.radians(offsets), dtype=torch.float32)
targets = target.expand_as(preds)

curves = {
    "exact 1 - IoU": [1 - rbox_iou(RBox(*p.tolist()), RBox(*target[0].tolist())) for p in preds],
    "decoded rIoU": (1 - diff_iou_rotated_2d(preds[None], targets[None])[0]).tolist(),
    "ProbIoU": probiou_loss_per_box(preds, targets).tolist(),
    "KFIoU": (1 - 3 * kfiou_overlap_ratio(preds, targets)).tolist(),
}
fig, ax = plt.subplots(figsize=(8, 4))
for name, ys in curves.items():
    ax.plot(offsets, ys, label=name, lw=2)
ax.set_xlabel("angle error (deg)"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
plt.show()
print("ProbIoU loss for a perfect box:", round(curves["ProbIoU"][0], 3), "(an eps floor, harmless for gradients)")

All four agree on the shape: zero error is best, 90° is worst, and 180° brings the loss back down because the box is the same again. Exact and decoded rIoU are identical; that is the point of the differentiable version. The Gaussian losses are smoother and never flat, which gives useful gradients even when boxes barely overlap.

## 2. Where Gaussians go blind: squares

A square's Gaussian is a circle, and a circle looks the same at every angle. ProbIoU therefore cannot tell a square at 0° from one at 45°, although their exact IoU is only 0.71. The effect fades as the box gets longer.

In [ ]:
print(f"{'aspect':>6s} {'exact IoU @45 deg':>18s} {'ProbIoU loss':>13s}")
for aspect in [1.0, 1.2, 1.5, 2.0, 4.0]:
    a = torch.tensor([[100.0, 100.0, 60.0 * aspect, 60.0, 0.0]])
    b = a.clone(); b[0, 4] = math.radians(45)
    print(f"{aspect:6.1f} {rbox_iou(RBox(*a[0].tolist()), RBox(*b[0].tolist())):18.2f} "
          f"{probiou_loss_per_box(b, a).item():13.3f}")

For near-square classes (storage tanks, roundabouts) the angle is meaningless anyway. For a long-but-not-very-long object it means weak angle supervision, one reason Oriented R-CNN keeps Smooth L1 as its main loss and uses ProbIoU only as an auxiliary term.

## 3. Fit one box by gradient descent

Optimise a single predicted box towards the target with Adam, once with ProbIoU and once with plain L1 on the five raw numbers.

In [ ]:
def fit(loss_fn, steps=600):
    pred = torch.tensor([[130.0, 80.0, 60.0, 70.0, 0.0]], requires_grad=True)
    opt = torch.optim.Adam([pred], lr=2.0)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    path = []
    for step in range(steps):
        opt.zero_grad()
        loss_fn(pred, target).sum().backward()
        opt.step()
        sched.step()
        with torch.no_grad():
            pred[:, 2:4].clamp_(min=1.0)
        if step % 100 == 0 or step == steps - 1:
            path.append(RBox(*pred.detach()[0].tolist()))
    return path


runs = {"ProbIoU": fit(probiou_loss_per_box), "L1 on raw numbers": fit(lambda p, t: (p - t).abs().sum(dim=1))}
goal = RBox(*target[0].tolist())
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (name, path) in zip(axes, runs.items()):
    for i, b in enumerate(path):
        xs, ys = zip(*b.corners(), b.corners()[0])
        ax.plot(xs, ys, color=plt.cm.viridis(i / (len(path) - 1)), lw=1.5)
    xs, ys = zip(*goal.corners(), goal.corners()[0])
    ax.plot(xs, ys, "r--", lw=2)
    end = path[-1]
    ax.set_title(f"{name}: w={end.width:.0f} h={end.height:.0f} angle={math.degrees(end.angle):.0f} deg\n"
                 f"exact IoU with target {rbox_iou(end, goal):.2f}", fontsize=10)
    ax.set_aspect("equal"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

same_box = torch.tensor([[100.0, 100.0, 40.0, 140.0, math.radians(120)]])  # target, other spelling
print(f"identical box, other spelling -> L1 {(same_box - target).abs().sum().item():.1f}, "
      f"ProbIoU {probiou_loss_per_box(same_box, target).item():.3f}")

Both runs end on the target rectangle, but not with the same numbers. L1 lands on the exact spelling of the target. ProbIoU lands on an *equivalent* spelling (width and height swapped, angle shifted by 90°) because it only sees the shape. The last line shows the flip side: L1 charges a large loss for a perfect box written differently, ProbIoU does not. That is why Gaussian losses sidestep the boundary problem, and why predictions go through `normalize_le90` before use.

## 4. Which detector uses which loss

Each recipe picks its box losses in the `model` section. The two-stage detectors configure their second-stage (RoI) head; the one-stage detectors configure their dense head.

In [ ]:
from oriented_det.train.config import TrainingExperimentConfig

RECIPES = ["oriented_rcnn/dota_le90_1x.json", "rotated_faster_rcnn/dota_le90_1x.json",
           "rotated_fcos/dota_le90_1x.json", "rotated_fcos/dota_le90_1x_l1_kfiou_aux.json",
           "rotated_retinanet/dota_le90_1x.json", "rotated_rtmdet/dota_le90_3x_s.json"]
print(f"{'recipe':44s} {'main box loss':14s} auxiliary")
for rel in RECIPES:
    if not (CONFIGS / rel).is_file():
        print(f"{rel:44s} (not in this oriented-det release)")
        continue
    cfg = TrainingExperimentConfig.load(CONFIGS / rel)
    m = cfg.model
    if cfg.model_type in ("oriented_rcnn", "rotated_faster_rcnn"):
        main, aux, w = m.roi_box_reg_main_loss_type, m.roi_box_reg_aux_loss_type, m.roi_box_reg_aux_weight
    else:
        main, aux, w = m.box_reg_loss_type, m.aux_loss_type, m.aux_loss_weight
    print(f"{rel:44s} {main:14s} {f'{aux} x{w}' if aux and w else '-'}")

| Detector | Main box loss | Auxiliary | Config keys |
|---|---|---|---|
| Oriented R-CNN | Smooth L1 on offsets | ProbIoU ×0.1 | `model.roi_box_reg_main_loss_type`, `model.roi_box_reg_aux_loss_type`, `model.roi_box_reg_aux_weight` |
| Rotated Faster R-CNN | ProbIoU | Smooth L1 ×0.1 | same keys |
| Rotated FCOS | decoded rIoU | — (the `_l1_kfiou_aux` variant: L1 + KFIoU ×0.1) | `model.box_reg_loss_type`, `model.aux_loss_type`, `model.aux_loss_weight` |
| Rotated RetinaNet | L1 | — | `model.box_reg_loss_type` |
| Rotated RTMDet | decoded rIoU (weight 2.0) | — | `model.box_reg_loss_type`, `model.box_reg_weight` |

ProbIoU is what let Rotated Faster R-CNN train on DOTA without MMCV's CUDA kernels: its 1× checkpoint scores **74.42%** on the official Task 1 server.

## What to try next

- `probiou_loss_per_box(..., mode="l2")` is the Hellinger-style variant. Add it to the sweep in section 1.
- Start the gradient-descent box at a 90° angle error and see which loss escapes the worst case fastest.
- Copy a recipe, switch `roi_box_reg_main_loss_type` between `"probiou"` and `"smooth_l1"`, and train both (lesson 07 shows how to patch and save a recipe).
- **Next:** [04 — Detector zoo and score thresholds](04_detector_zoo_thresholds.ipynb).